# Scratch Book

## Scoring by origin for 2024-25 full season stats

In [185]:
import os
import sys
from pathlib import Path
import pandas as pd
import regex as re

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

project_root = base_dir.parent


# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scoring_origins"

# ======= IMPORT CONFIG =======
import config  # now you can import config.py


school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

In [186]:
## Get New Copy of 2024_25 Roster using old scraper



In [187]:
## Connect to DB with 2024-25 data
import sqlalchemy
from sqlalchemy import create_engine

db_path = data_folder / "db" / "2024_2025_Season_Final.db"
engine = create_engine(f"sqlite:///{db_path}")
connection = engine.connect()

# print list of tables in the database
inspector = sqlalchemy.inspect(engine)
tables = inspector.get_table_names()
print(tables)

# # extract Master Roster and save as a csv ### NOT USING THE ONE IN DB TRYING ONE FROM MARCH SCRAPE
# master_roster_df = pd.read_sql("SELECT * FROM master_roster", connection)
# master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
# master_roster_df.to_csv(master_roster_file, index=False)

### READ master_roster from NEW SCRAPE CSV
master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
master_roster_df = pd.read_csv(master_roster_file)

['advanced_metrics', 'game_details', 'goalie_stats', 'line_chart', 'linescore', 'master_roster', 'penalty_summary', 'player_stats', 'player_stats_ytd', 'scoring_summary']


In [188]:
# get ytd_stats table
ytd_stats_df = pd.read_sql("SELECT * FROM player_stats_ytd", connection)
ytd_stats_file = roster_folder / "2024_25_Season Stats.csv"
ytd_stats_df.to_csv(ytd_stats_file, index=False)

## Find a player on rensselaer, last name Caron. find player in ytd_stats_df that contains 'Caron' in Clean_Player column
caron_player = ytd_stats_df[ytd_stats_df['Clean_Player'].str.contains('Caron', na=False)]

## Simplify player name from Félix Caron to Felix Caron for matching
caron_player['Clean_Player'] = caron_player['Clean_Player'].str.replace('é', 'e')

print(caron_player)

       Clean_Player               Team  G   A  Pts  plus_minus  Sh  TOI_sec  \
574     Felix Caron         Rensselaer  6  15   21         -10  53  32116.0   
1099  Mathieu Caron  Boston University  0   3    3           0   0      0.0   

      PIM  FOW  FOL  Games_Played        FO%       TOI  
574    62  4.0  2.0            35  66.666667  08:55:16  
1099    0  0.0  0.0            21        NaN  00:00:00  


C:\Users\jbanc\AppData\Local\Temp\ipykernel_26792\2447087082.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caron_player['Clean_Player'] = caron_player['Clean_Player'].str.replace('é', 'e')


In [189]:
## # Combine First and last name in roster_df to match player_ytd_df
roster_df = master_roster_df
# Clean white space from names
roster_df["First_Name"] = roster_df["First_Name"].str.strip()
roster_df["Last_Name"] = roster_df["Last_Name"].str.strip()
roster_df["Clean_Player"] = roster_df["First_Name"] + " " + roster_df["Last_Name"]
# Strip any leading/trailing whitespace
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
# Rename Current Team to match player_ytd_df
roster_df = roster_df.rename(columns={"Current Team": "Team"})
# Reorder columns for easier viewing
#Order of columns
# ["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']
roster_df = roster_df[["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']]    




roster_df.head()

## State_Province Value Counts
check_states = roster_df["State_Province"].value_counts(dropna=False)
check_states


State_Province
Minnesota                248
Ontario                  183
Michigan                 118
Massachusetts            116
New York                 115
                        ... 
South Carolina             1
Montana                    1
Northwest Territories      1
Austria                    1
Nebraska                   1
Name: count, Length: 72, dtype: int64

### Roster Fixes and Mods

In [190]:
### Roster Modifications to make up for incomplete data ###

### Replace bad values in State_Province column
state_province_corrections = {"Okla.": "Oklahoma", "D.C": "District of Columbia"
}
# Apply state/province corrections
roster_df["State_Province"] = roster_df["State_Province"].replace(state_province_corrections)

### Replace country abbreviations with full country names
country_corrections = {"CYM": "Cayman Islands", "JPN": "Japan", "RUS": "Russia"
}
# Apply country corrections
roster_df["Country"] = roster_df["Country"].replace(country_corrections)

In [191]:
player_ytd_df = ytd_stats_df

# Make sure name and team columns are stripped of punctuation, strange characters, and whitespace
player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.strip()
player_ytd_df["Team"] = player_ytd_df["Team"].str.strip()
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
roster_df['Clean_Player'] = roster_df['Clean_Player'].str.replace(u'\xa0', u' ').str.strip()  # Cleanup player name
roster_df["Team"] = roster_df["Team"].str.strip()
# # Change American Int'l to American Intl in Team Columns
player_ytd_df["Team"] = player_ytd_df["Team"].replace("American Int'l", "American Intl")
roster_df["Team"] = roster_df["Team"].replace("American Int'l", "American Intl")
# Remove any hyphens & periods from team names, replace with ' '
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
roster_df["Team"] = roster_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
# Remove any hyphens, periods, ect from Clean Player names to match
# player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.replace(r'[^\w\s]', '', regex=True)
# roster_df["Clean_Player"] = roster_df["Clean_Player"].str.replace(r'[^\w\s]', '', regex=True)
# QUICK FIX - Standardize team names with double spaces
# If team name column has double spaces, replace with single space
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace('  ', ' ', regex=False)
roster_df["Team"] = roster_df["Team"].str.replace('  ', ' ', regex=False)

## Merge the two DataFrames on Clean_Player and Team
merged_df = pd.merge(
    player_ytd_df,
    roster_df,
    left_on=["Clean_Player", "Team"],
    right_on=["Clean_Player", "Team"],
    how="left"
)

# Shape and info of merged DataFrame
print("Merged DataFrame shape:", merged_df.shape)
# print(merged_df.columns)
# print(merged_df.head())
# merged_df.info()



Merged DataFrame shape: (1753, 32)


### Insert Last Team and League from manually researched ones

In [192]:
## Load csv file with player origins needed in corrections
corrections_file = roster_folder / "2024_25_player_origin_corrections.csv"
corrections_df = pd.read_csv(corrections_file)

corrections_df.head()

# Use Clean_Player as key, create dictionary with Last_Team and League
corrections = {}
for index, row in corrections_df.iterrows():
    corrections[row['Clean_Player']] = {
        'Last_Team': row['Last Team'],
        'League': row['League']
    }

print(corrections)

## Apply corrections to roster_df
def apply_corrections(row):
    player_name = row['Clean_Player']
    if player_name in corrections:
        row['Last Team'] = corrections[player_name]['Last_Team']
        row['League'] = corrections[player_name]['League']
    return row

merged_df = merged_df.apply(apply_corrections, axis=1)

{'Alexander Lundman': {'Last_Team': 'Bemidji State', 'League': 'NCAA-DI'}, 'Alexander Tell': {'Last_Team': 'Linkoping HC J20 "A"', 'League': 'Europe'}, 'Anthony Galante': {'Last_Team': 'Johnstown', 'League': 'NAHL'}, 'Artem Buzoverya': {'Last_Team': 'Hobart', 'League': 'DIII'}, 'Ben Ivey': {'Last_Team': 'Amarillo', 'League': 'NAHL'}, 'Brady Cleveland': {'Last_Team': 'Wisconsin', 'League': 'NCAA-DI'}, 'Brian Nicholas': {'Last_Team': 'Souix City', 'League': 'USHL'}, 'Cade Christenson': {'Last_Team': 'Sherwood Park', 'League': 'BCHL'}, 'Caleb Price': {'Last_Team': 'Lindenwood', 'League': 'NCAA-DI'}, 'Cam Mannion': {'Last_Team': 'Thayer Academy', 'League': 'PREP'}, 'Casper Nassen': {'Last_Team': 'Frölunda HC J20', 'League': 'Europe'}, 'Cayden Casey': {'Last_Team': 'Minot', 'League': 'NAHL'}, 'Charles Banquier': {'Last_Team': 'Prince George', 'League': 'BCHL'}, 'Charlie Leddy': {'Last_Team': 'Boston College', 'League': 'NCAA-DI'}, 'Chris Pelosi': {'Last_Team': 'Sioux Fall', 'League': 'USHL'

In [193]:
merged_df.head()

,Clean_Player,Team,G,A,Pts,plus_minus,Sh,TOI_sec,PIM,FOW,...,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country
0,A.J. Hodges,Bentley,9,9,18,7,78,28504.0,8,19.0,...,"Littleton, Colo.",72.0,NaN,NaN,NaN,Michigan State,B10,Littleton,Colorado,USA
1,A.J. Macaulay,Bemidji State,0,2,2,-5,17,22475.0,4,0.0,...,"Bonnyville, Alb.",69.0,NaN,NaN,NaN,Alaska,D-I Ind.,Bonnyville,Alberta,Canada
2,AJ Casperson,Long Island,0,0,0,0,1,249.0,0,0.0,...,"Flower Mound, Texas",74.0,NaN,NaN,NaN,Janesville,NAHL,Flower Mound,Texas,USA
3,Aaron Bohlinger,Quinnipiac,3,11,14,11,34,40704.0,4,0.0,...,"Walden, N.Y.",69.0,NaN,NaN,NaN,Waterloo,USHL,Walden,New York,USA
4,Aaron Grounds,Long Island,0,0,0,0,0,544.0,0,0.0,...,"Jamestown, N.D.",74.0,NaN,NaN,NaN,American Int'l,AHA,Jamestown,North Dakota,USA


In [194]:
### Print dataframe stats for to keep track of filtering steps
original_count = merged_df.shape[0]
print("Merged DataFrame shape before filtering:", merged_df.shape)

### Filter out players with no TOI and goalies
# Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
# Remove any Rows where TOI_sec is 0 or NaN - These are goalies or players with no time on ice
merged_df = merged_df[(merged_df["TOI_sec"] > 0) & (~merged_df["TOI_sec"].isna())]
first_step_count = merged_df.shape[0]

## Check the shape after filtering
print("Merged DataFrame shape after filtering TOI_sec > 0:", merged_df.shape)
# Number of players removed
print("Players removed after filtering TOI_sec > 0:", original_count - first_step_count)

### DO NOT NEED TO FILTER FOR GOALIES BECAUSE PLAYER_YTD_STATS TABLE ONLY HAS TOI FOR SKATERS
# Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)
# Strip any whitespace from Position column
# merged_df["Position"] = merged_df["Position"].str.strip()
# merged_df = merged_df[merged_df["Position"] != "Goaltenders"]
# no_goalie_count = merged_df.shape[0]
## Check the shape after filtering
# print("Merged DataFrame shape after filtering out Goalies:", merged_df.shape)
# # Number of players removed
# print("Players removed by filtering out Goalies:", first_step_count - no_goalie_count)

Merged DataFrame shape before filtering: (1753, 32)
Merged DataFrame shape after filtering TOI_sec > 0: (1581, 32)
Players removed after filtering TOI_sec > 0: 172


In [195]:
### Reusing League and Team Classification function and libraries from team_construction_visual_workbook import classify_previous_team, classify_previous_league

# ----------------------------
# 1) Rename merged_df to df for easier to fit in with existing code
# -----------------------------
df = merged_df.copy()

# -----------------------------
# 2) Classification helpers
# -----------------------------
def _norm_set(strings):
    return { _norm(s) for s in strings }

def _norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().upper()
    s = re.sub(r"[.\u2010-\u2015\-–—]+", " ", s)  # unify hyphen-like chars to space
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _norm_set(strings):
    return { _norm(s) for s in strings }

NTDP_TEAM_HINTS_RAW = (
    "USA U 18", "US U 18", "USA U18", "US U18",
    "USA U 17", "US U 17", "USA U17", "US U17",
    "NTDP", "USNTDP", "US NATIONAL TEAM", "US DEV PROGRAM", "US DEVELOPMENT PROGRAM"
)
NTDP_LEAGUE_HINTS_RAW = ("NTDP", "USNTDP")

D1_CONFS_RAW = {"ECAC","CCHA","NCHC","HEA","B10","AHA","INDEPENDENTS","D I IND","NCAA", "NCAA-DI"}

CJHL_LEAGUES_RAW = {"BCHL","AJHL","SJHL","OJHL", "MJHL", "CCHL","MHL"}

US_TIER1_RAW = {"USHL"}
US_TIER2_RAW = {"NAHL","NCDC"}
US_OTHER_RAW  = {"USPHL","NA3HL", "PREP", "PHC", "USHS", "CISAA", "DIII", "D-III", "EHL", "ACHA"}

CHL_RAW = {"OHL","WHL","QMJHL"}

EURO_HINTS_RAW = {
    "J20 NATIONELL","J18 REGION","U20 SM SARJA","U18","U20","SM SARJA",
    "SHL","ICEHL","ALPSHL","LIIGA","MHL RUSSIA","KHL, ICEHL","KHL","DEL", "EC-KAC", 
    "Oberliga", "Europe", "SWE", "Sweden"
}

EURO_TEAM_HINTS_RAW = {
    "KalPa U20", "Frölunda HC", "Malmo", "Jokerit U20", "U20 SM Sarja-Pelicans",
    "Tappara J20", "Djurgårdens IF", "Leksands IF", "Mora IK J20"
}

RUSSIAN_MHL_TEAM_HINTS_RAW = (
    "KRASNAYA","LOKO","MOSKVA","MOSCOW","ST PETERSBURG","SKA","LOKOMOTIV",
    "DMITROV","CHELYABINSK","OMSK","NOVOSIBIRSK","MAGNITOGORSK","NIZHNY","YAROSLAVL",
    "Karlskrona HK"
)

# Normalize them
NTDP_TEAM_HINTS       = _norm_set(NTDP_TEAM_HINTS_RAW)
NTDP_LEAGUE_HINTS     = _norm_set(NTDP_LEAGUE_HINTS_RAW)
D1_CONFS              = _norm_set(D1_CONFS_RAW)
CJHL_LEAGUES          = _norm_set(CJHL_LEAGUES_RAW)
US_TIER1              = _norm_set(US_TIER1_RAW)
US_TIER2              = _norm_set(US_TIER2_RAW)
US_OTHER              = _norm_set(US_OTHER_RAW)
CHL                   = _norm_set(CHL_RAW)
EURO_HINTS            = _norm_set(EURO_HINTS_RAW)
EURO_TEAM_HINTS       = _norm_set(EURO_TEAM_HINTS_RAW)
RUSSIAN_MHL_TEAM_HINTS = _norm_set(RUSSIAN_MHL_TEAM_HINTS_RAW)

PRO_HINTS = ("AHL","ECHL")

BIN_ORDER = [
    "NTDP",
    "USHL (non‑NTDP)",
    "NAHL/NCDC",
    "US (DIII/Prep/Other)",
    "CHL (Major Junior)",
    "CJHL (Canadian Jr A)",
    "U SPORTS",
    "Europe",
    "NCAA D1 Transfers",
    "Pro (AHL/ECHL/Other)",
    "Other/Various/Unknown"
    
]

COLOR_MAP = {
    "NTDP": "#0057B8",
    "USHL (non‑NTDP)": "#1E90FF",
    "NAHL/NCDC": "#63B8FF",
    "US (DIII/Prep/Other)": "#B0E2FF",
    "CHL (Major Junior)": "#B22222",
    "CJHL (Canadian Jr A)": "#FF7F7F",
    "U SPORTS": "#FADBD8",
    "Europe": "#F0E130",
    "NCAA D1 Transfers": "#696969",
    "Pro (AHL/ECHL/Other)": "#000000",
    "Other/Various/Unknown": "#A9A9A9"
}

def classify_prev_bin(last_team: str, league: str) -> str:
    t = _norm(last_team)
    l = _norm(league)

    # NTDP carve-out first
    if any(h in t for h in NTDP_TEAM_HINTS) or any(h == l for h in NTDP_LEAGUE_HINTS):
        return "NTDP"

    # D3/Prep -> Other
    if l in US_OTHER:
        return "US (DIII/Prep/Other)"

    # NCAA D1 transfers
    if (l in D1_CONFS) or ("NCAA" in l and l != ""):
        return "NCAA D1 Transfers"

    # U SPORTS
    if l in {"USPORTS", "U SPORTS"}:
        return "U SPORTS"

    # CHL
    if l in CHL:
        return "CHL (Major Junior)"

    # MHL ambiguity
    if l == "MHL":
        if any(k in t for k in RUSSIAN_MHL_TEAM_HINTS):
            return "Europe"
        else:
            return "CJHL (Canadian Jr A)"

    # CJHL
    if l in CJHL_LEAGUES:
        return "CJHL (Canadian Jr A)"



    # US juniors
    if l in US_TIER1:
        return "USHL (non‑NTDP)"
    if l in US_TIER2:
        return "NAHL/NCDC"
    
    # Pro leagues
    if any(h in l for h in PRO_HINTS):
        return "Pro (AHL/ECHL/Other)"

    # Europe consolidated
    if l in EURO_HINTS:
        return "Europe"
    if t in EURO_TEAM_HINTS:
        return "Europe"




    # Fallbacks/Unknown
    # if l == "" or pd.isna(league):
    #     return "Other/Various/Unknown"
    
    if "U SPORTS" in t:
        return "U SPORTS"
    if "IF Sundsvall" in t:  # edge case
        return "Europe"


    return "Other/Various/Unknown"


# Apply classification to the raw roster
df["Prev_League_Bin"] = df.apply(lambda r: classify_prev_bin(r.get("Last Team", np.nan),
                                                             r.get("League", np.nan)), axis=1)
df["Prev_League_Bin"] = pd.Categorical(df["Prev_League_Bin"], categories=BIN_ORDER, ordered=True)

# Save classified to temp_folder
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# out_file = os.path.join(temp_folder, f"roster_classified_{timestamp}.csv")
# df.to_csv(out_file, index=False)


# show quick counts
overall_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

print("Overall Counts by Prev_League_Bin")
print(overall_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":"Count"}))

Overall Counts by Prev_League_Bin
                    Count  count
0                    NTDP     45
1         USHL (non‑NTDP)    443
2               NAHL/NCDC    276
3    US (DIII/Prep/Other)     16
4      CHL (Major Junior)      0
5    CJHL (Canadian Jr A)    364
6                U SPORTS      0
7                  Europe     34
8       NCAA D1 Transfers    402
9    Pro (AHL/ECHL/Other)      1
10  Other/Various/Unknown      0


In [196]:
## Show The Other/Various/Unknown players
other_unknown_df = df[df["Prev_League_Bin"] == "Other/Various/Unknown"]
other_unknown_df = other_unknown_df[["Clean_Player", "Team", "Last Team", "League", "Prev_League_Bin" ]]
other_unknown_df

,Clean_Player,Team,Last Team,League,Prev_League_Bin


In [197]:
### Output 2024_25 merged_df with classifications to csv
output_file = roster_folder / "2024_25_Player_Stats_with_Origins_NEW.csv"
df.to_csv(output_file, index=False)

### Create Library of Color Map and logos

In [198]:
# # create library of team color mappings and logos to use in plots

# import os
# import pandas as pd

# # path to TEMP folder
# temp_folder = os.path.join(os.getcwd(), '..', 'TEMP')
# # Data folder
# data_folder = os.path.join(os.getcwd(), '..', 'data')
# # print(os.listdir(data_folder)) # Print List of files in data folder

# # path to School Info folder in data
# school_info_folder = os.path.join(os.getcwd(), data_folder, 'school_info')

# # Image folders
# img_folder = os.path.join(os.getcwd(), '..', '..', 'images') # base image folder
# # Logo folder
# logo_folder = os.path.join(os.getcwd(), '..', '..', 'images', 'logos')
# # print(os.listdir(logo_folder)) # Print List of files in logo folder

# # Background folder
# background_folder = os.path.join(img_folder, 'background')
# # print(os.listdir(background_folder)) # Print List of files in background folder

# # Plot Output folder
# plot_folder = os.path.join(os.getcwd(), '..', '..', 'TEMP', 'IMAGE')



# ################################################################################


# # Path to school info table (csv)
# school_info_file = os.path.join(school_info_folder, 'arena_school_info.csv')
# school_info_df = pd.read_csv(school_info_file)

In [199]:
school_info_df

,Team,Arena,Capacity,Sheet_length,Sheet_width,School,Latitude,Longitude,hex1,hex2,hex3,simp_color,logo_abv,abv,ncaa_name,ncaa_data_alts,eliteprospects_url
0,Air Force,Cadet Ice Arena,2470,200,85,Air Force,39.013739,-104.883727,3087,8a8d8f,NaN,NaN,afa,Air Force,Air Force,"AIRFOR, Air Force",https://www.eliteprospects.com/team/2453/air-f...
1,Alaska,Carlson Center,4595,200,100,Alaska,64.842124,-147.763841,236192,ffcd00,NaN,NaN,akf,Alaska,Alas Fairbanks,"AK FBK, Alas. Fairbanks",https://www.eliteprospects.com/team/2071/univ....
2,Alaska Anchorage,Avis Alaska Sports Complex,800,200,85,Alaska-Anchorage,61.205536,-149.872737,00583d,ffc425,NaN,NaN,aka,UAA,Alas Anchorage,"AK ANC, Alas. Anchorage",https://www.eliteprospects.com/team/1915/univ....
3,American Intl,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...
4,American Int'l,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,Union,Achilles Center,2504,200,85,Union,42.818004,-73.924824,9a0000,ffffff,NaN,NaN,uni,Union,Union NY,"UNION, Union (NY)",https://www.eliteprospects.com/team/1366/union...
65,Vermont,Gutterson Fieldhouse,4003,200,90,Vermont,44.469526,-73.193317,154734,8b5b29,NaN,NaN,ver,UVM,Vermont,"VERMNT, Vermont",https://www.eliteprospects.com/team/710/univ.-...
66,Western Michigan,Lawson Ice Arena,3667,200,85,Western Michigan,42.284525,-85.610224,6c4023,b5a167,NaN,NaN,wmu,WMU,Western Mich,"W MICH, Western Mich.",https://www.eliteprospects.com/team/1250/weste...
67,Wisconsin,Kohl Center,15237,200,97,Wisconsin,43.069489,-89.396972,c5050c,ffffff,NaN,red,wis,Wisconsin,Wisconsin,"WISC, Wisconsin",https://www.eliteprospects.com/team/452/univ.-...


In [200]:
# # Preprocess school info dataframe to create color and logo mappings
# # hex1 and hex2 columns need to be normalized to ensure they are 6-character hex codes
# def normalize_hex_color(hex_color):
#     if pd.isna(hex_color):
#         return None
#     hex_color = str(hex_color).lstrip('#')
#     if len(hex_color) == 3:
#         hex_color = ''.join([c*2 for c in hex_color])
#     return f'#{hex_color.zfill(6)}'
# school_info_df['hex1'] = school_info_df['hex1'].apply(normalize_hex_color)
# school_info_df['hex2'] = school_info_df['hex2'].apply(normalize_hex_color)
# school_info_df['hex3'] = school_info_df['hex3'].apply(normalize_hex_color)

# # Create color mapping dictionary
# team_color_map = {}
# for _, row in school_info_df.iterrows():
#     team_name = row['Team']
#     color1 = row['hex1'] if pd.notna(row['hex1']) else '#000000'  # default to black
#     color2 = row['hex2'] if pd.notna(row['hex2']) else '#FFFFFF'  # default to white
#     color3 = row['hex3'] if pd.notna(row['hex3']) else '#CCCCCC'  # default to gray
#     team_color_map[team_name] = (color1, color2, color3)

# # Create logo mapping dictionary
# team_logo_map = {}
# for _, row in school_info_df.iterrows():
#     team_name = row['Team']
#     logo_path = row['logo_abv'] if pd.notna(row['logo_abv']) else None
#     team_logo_map[team_name] = logo_path


In [201]:
team_color_map

NameError: name 'team_color_map' is not defined